# Figures Calcul 001

In [10]:
# import necessary libraries
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import re
from typing import Dict, List, Tuple, Any
from matplotlib.patches import Circle

In [11]:
# File paths
data_path = "../data/001MoDe_R1.csv"
marker_path = "../data/001MoDe_R1.marker.csv"

### 1. Parse Header From Data File

In [12]:
def parse_header(data_path: str) -> Dict[str, Any]:
    """Parses the header from the first non-empty line of the data file.
    
    - Objective:
            Extract key–value metadata stored in the first non-empty line of a CSV-like
            file. The function handles multiple formats such as `key=value`,
            `key:value`, or `key value`.

    - Inputs:
            data_path (str): Path to the data file from which the header should be read.

    - Outputs:
            Dict[str, Any]: A dictionary containing the parsed header fields.  
            Numeric values are converted to float when possible; all other values  
            remain as strings. Returns an empty dictionary if the file is not found  
             or if no header line is detected.
    """
    header: Dict[str, Any] = {}
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            first_line = ""
            for line in f:
                stripped_line = line.strip()
                if stripped_line:
                    first_line = stripped_line
                    break

            if not first_line:
                return header

            for part in first_line.split(";"):
                part = part.strip()
                if not part:
                    continue

                if "=" in part:
                    k, v = part.split("=", 1)
                elif ":" in part:
                    k, v = part.split(":", 1)
                else:
                    tokens = part.split()
                    k = tokens[0]
                    v = tokens[1] if len(tokens) > 1 else ""

                k, v = k.strip(), v.strip()
                try:
                    header[k] = float(v)
                except (ValueError, TypeError):
                    header[k] = v

    except FileNotFoundError:
        print(f"Error: Data file not found at {data_path}")
        return {}

    return header

### 2. Parse Markers From Marker File

In [13]:
def parse_markers(marker_path: str) -> Dict[str, Any]:
    """Parse a marker file to extract recording start and end timestamps.

    - Objective:
            Read a marker log file and identify the timestamps corresponding to 
            "DoRecord" and "DoPause" events. These markers define the start and end 
            of recorded segments. The function also provides the full list of markers 
            and matches each start event to its corresponding end event.

    - Inputs:
            marker_path (str): Path to the marker file. The file is expected to contain
            lines formatted with at least two comma-separated fields, where the
            second field is a timestamp.

    - Outputs:
            Dict[str, Any]: A dictionary containing:
                "start_ts" (List[int]): Sorted list of detected start timestamps.
                "end_ts" (List[int]): Sorted list of detected end timestamps.
                "pairs" (List[Tuple[int, int]]): List of matched (start, end) timestamp pairs.
                "all_markers" (List[Tuple[int, str]]): All markers found, each as (timestamp, raw_line).
      
      Raises informative exceptions if the marker file is missing or if no valid
      start/end markers can be extracted.
    """
    record_starts_raw: List[int] = []
    record_ends_raw: List[int] = []
    all_markers: List[Tuple[int, str]] = []

    try:
        with open(marker_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        raise FileNotFoundError(f"Marker file not found at {marker_path}")

    for line in lines:
        parts = line.strip().split(",")
        if len(parts) < 2:
            continue

        ts_str = parts[1].strip().replace(" ", "")
        try:
            ts = int(ts_str)
        except ValueError:
            continue

        all_markers.append((ts, line.strip()))

        if "DoCycleChange:DoRecord" in line:
            record_starts_raw.append(ts)
        elif "DoCycleChange:DoPause" in line:
            record_ends_raw.append(ts)

    if not record_starts_raw:
        raise ValueError("No 'DoRecord' events detected in the marker file.")
    if not record_ends_raw:
        raise ValueError("No 'DoPause' events detected in the marker file.")

    record_starts_raw.sort()
    record_ends_raw.sort()

    pairs: List[Tuple[int, int]] = []
    end_idx = 0
    for start_ts in record_starts_raw:
        while end_idx < len(record_ends_raw) and record_ends_raw[end_idx] < start_ts:
            end_idx += 1
        if end_idx < len(record_ends_raw):
            pairs.append((start_ts, record_ends_raw[end_idx]))
            end_idx += 1
        else:
            break

    if not pairs:
        raise RuntimeError("No valid start/end pairs found in markers.")

    return {
        "start_ts": [p[0] for p in pairs],
        "end_ts": [p[1] for p in pairs],
        "pairs": pairs,
        "all_markers": all_markers,
    }